In [ ]:
import pandas as pd, numpy as np

df=pd.read_excel('00权重数据提取.xlsx')

def determine_hcv(row):
    if row['LBDHCI'] in [1, 4]:
        return 1
    elif pd.notna(row['LBDHCI']):
        return 0
    elif row['LBDHCV'] == 1:
        return 1
    elif pd.notna(row['LBDHCV']) and row['LBDHCV'] not in [1,5]:
        return 0
    elif row['LBXHCR'] == 1:
        return 1
    elif pd.notna(row['LBXHCR']):
        return 0
    elif row['SSHCVRNA'] == 1:
        return 1
    elif pd.notna(row['SSHCVRNA']):
        return 0
    else:
        return pd.NA

df['HCV'] = df.apply(determine_hcv, axis=1)
# 删除不需要的列（去重）
df = df.drop(columns=list(set([ 'LBDHCI', 'LBDHCV', 'LBXHCR', 'SSHCVRNA'])))

# 重命名列
df = df.rename(columns={
    'LBDHBG': 'HBsAg',
    'LBXHBS': 'HBsAb',
    'LBDHEM': 'HBeAb',
    'LBXHBC': 'HBcAb',
    '协变量PIR': 'PIR',
    '协变量_吸烟状态': 'Smoking',
    '协变量婚姻': 'Marital_status',
    '协变量年龄': 'Age',
    '协变量性别': 'Gender',
    '协变量文化_全人群': 'Education',
    '协变量种族': 'Race',
    '协变量饮酒': 'Alcohol'
})

# 修改 Marital_status，保留缺失值
df['Marital_status'] = df['Marital_status'].where(df['Marital_status'].isna(), 
                             df['Marital_status'].apply(lambda x: 1 if x in [1, 6] else 0))

# 年龄分组（18–29, 30–44, 45–64, 65+），保留空值
df['Age_cat'] = pd.cut(
    df['Age'],
    bins=[17, 29, 44, 64, np.inf],
    labels=[0, 1, 2, 3]
)
df['Race'] = df['Race']-1
df['Education']=df['Education']-1
df['HBsAg']=2-df['HBsAg']
df['Gender']=2-df['Gender']

# 根据 PIR 划分为三类PIR 分类：<=1.3 → 0，1.3~3.5 → 1，>3.5 → 2，保留空值
#分类依据：https://translational-medicine.biomedcentral.com/articles/10.1186/s12967-024-04969-3
df['PIR_cat'] = df['PIR'].apply(
    lambda x: np.nan if pd.isna(x) else (0 if x <=1.3  else (1 if x <= 3.5 else 2))
)

# 将指定列排在前面，其余列顺序不变
priority_cols = [
    'HCV', 'HBsAg', 'HBsAb', 'HBeAb', 'HBcAb',
    'Age', 'Age_cat', 'Gender', 'Race', 'Education',
    'Marital_status', 'PIR', 'PIR_cat', 'Smoking', 'Alcohol',
]

# 保证优先列在 df 中存在，然后排列其余列
existing_priority = [col for col in priority_cols if col in df.columns]
remaining_cols = [col for col in df.columns if col not in existing_priority]
df = df[existing_priority + remaining_cols]
# 原始行数
original_len = len(df)

# 第一步：删除 HCV 为空的行
len_before = len(df)
df = df.dropna(subset=['HCV'])
print(f"因 HCV 缺失删除了 {len_before - len(df)} 行")
df.to_excel("01data_重命名_划分Age_PIR.xlsx", index=False)

因 HCV 缺失删除了 30 行


FIB 2.67
https://www.cghjournal.org/article/S1542-3565(25)00081-3/fulltext
https://onlinelibrary.wiley.com/doi/full/10.1111/apt.70286?saml_referrer

FIB-4 1.45,3.25
https://onlinelibrary.wiley.com/doi/full/10.1002/hep.21178


In [2]:
import pandas as pd
import numpy as np

# 读取原始数据
df = pd.read_excel('04data_处理NAFLD计算LC9删除调整重命名.xlsx')

# 原始行数
original_len = len(df)

# 第一步：删除 Alcohol 为空的行
len_before = len(df)
df = df.dropna(subset=['Alcohol'])
print(f"因 Alcohol 缺失删除了 {len_before - len(df)} 行")

# 替换列的值
df['HCV'] = df['HCV'].replace({0: 'No', 1: 'Yes'})
df['HBsAg'] = df['HBsAg'].replace({1: 'Positive', 2: 'Negative'})
df['Age_cat'] = df['Age_cat'].replace({
    0: '18–29',
    1: '30–44',
    2: '45–64',
    3: '65+'
})
df['Gender'] = df['Gender'].replace({1: 'Male', 2: 'Female'})
df['Race'] = df['Race'].replace({
    1: 'Mexican American',
    2: 'Other Hispanic',
    3: 'Non-Hispanic White',
    4: 'Non-Hispanic Black',
    5: 'Others'
})
df['Education'] = df['Education'].replace({
    1: 'Less than high school',
    2: 'High school or equivalent',
    3: 'College or above'
})
df['Marital_status'] = df['Marital_status'].replace({
    1: 'Married/Living with partner',
    0: 'Widowed/Divorced/Separated/Never married'
})
df['PIR_cat'] = df['PIR_cat'].replace({
    0: 'Low',
    1: 'Medium',
    2: 'High'
})
df['Alcohol'] = df['Alcohol'].replace({
    0: 'No',
    1: 'Yes'
})

# 保存清洗后的数据
df.to_excel("05data_删除Alcohol空值_协变量替换字符串.xlsx", index=False)

因 Alcohol 缺失删除了 222 行


In [ ]:
import pandas as pd
import numpy as np

# 读取原始数据
df = pd.read_excel('09data_四分LC9.xlsx')

# 原始行数
original_len = len(df)

# 替换列的值（字符串 -> 数字）
df['HCV'] = df['HCV'].replace({'No': 0, 'Yes': 1})
df['HBsAg'] = df['HBsAg'].replace({'Negative': 0, 'Positive': 1})
df['Age_cat'] = df['Age_cat'].replace({
    '18–29': 0,
    '30–44': 1,
    '45–64': 2,
    '≥65': 3
})
df['Gender'] = df['Gender'].replace({'Female': 0, 'Male': 1})
df['Race'] = df['Race'].replace({
    'Mexican American': 0,
    'Other Hispanic': 1,
    'Non-Hispanic White': 2,
    'Non-Hispanic Black': 3,
    'Others': 4
})
df['Education'] = df['Education'].replace({
    'Less than high school': 0,
    'High school or equivalent': 1,
    'College or above': 2
})
df['Marital_status'] = df['Marital_status'].replace({
    'Widowed/Divorced/Separated/Never married': 0,
    'Married/Living with partner': 1
})
df['PIR_cat'] = df['PIR_cat'].replace({
    'Low': 0,
    'Medium': 1,
    'High': 2
})
df['Alcohol'] = df['Alcohol'].replace({
    'No': 0,
    'Yes': 1
})

# 保存清洗后的数据
df.to_excel("10data_协变量字符串替换成数字.xlsx", index=False)